<a href="https://colab.research.google.com/github/k-bridget/ML_Pipeline_Formative2-Group22/blob/main/04_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mount Drive

In [26]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Model Creation

In [27]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report

# XGBoost support
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

os.makedirs("models", exist_ok=True)


def evaluate_model(title, model, X_train, X_test, y_train, y_test, class_names):
    """Fit model and compute Accuracy, Macro F1-Score, and Log Loss."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    loss = log_loss(y_test, y_proba, labels=np.arange(len(class_names)))

    print(f"\n{'='*60}")
    print(f"MODEL: {title}")
    print(f"{'='*60}")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-Score : {f1:.4f} (Macro)")
    print(f"Log Loss : {loss:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))
    return acc, f1, loss

#Facial Recognition Model


In [28]:
img_df = pd.read_csv("/content/drive/MyDrive/image_features.csv")

face_encoder = LabelEncoder()
y_face = face_encoder.fit_transform(img_df["person"])

face_drop_cols = ["person", "expression", "augmentation", "source_file"]
X_face = img_df.drop(columns=[c for c in face_drop_cols if c in img_df.columns])

# Expression-based split: train on neutral + smiling (30 rows), test on unseen surprised (15 rows)
face_train_mask = img_df["expression"].isin(["neutral", "smiling"])
face_test_mask = img_df["expression"] == "surprised"

X_face_train, y_face_train = X_face[face_train_mask], y_face[face_train_mask]
X_face_test, y_face_test = X_face[face_test_mask], y_face[face_test_mask]

face_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
])

evaluate_model(
    "1. Facial Recognition Model (Logistic Regression)",
    face_model,
    X_face_train, X_face_test,
    y_face_train, y_face_test,
    face_encoder.classes_,
)

# Refit on all 45 samples for final CLI deployment
face_model.fit(X_face, y_face)
joblib.dump(face_model, "models/face_model.joblib")
joblib.dump(face_encoder, "models/face_encoder.joblib")
joblib.dump(list(X_face.columns), "models/face_features.joblib")


MODEL: 1. Facial Recognition Model (Logistic Regression)
Accuracy : 1.0000
F1-Score : 1.0000 (Macro)
Log Loss : 0.0120

Classification Report:
              precision    recall  f1-score   support

     Bridget       1.00      1.00      1.00         5
       Merci       1.00      1.00      1.00         5
        Yves       1.00      1.00      1.00         5

    accuracy                           1.00        15
   macro avg       1.00      1.00      1.00        15
weighted avg       1.00      1.00      1.00        15



['models/face_features.joblib']

- **Metrics:** Accuracy = **1.0000 (100.00%)** | Macro F1-Score = **1.0000** | Log Loss = **0.0120**
Training neutral and smiling images and testing unseen surprised expressions, the model correctly identified all 15 test images.The Log Loss is almost zero(0.0120) This indicates that the extracted facial features separate the three members with high certainity, even when facial geometry changes.

#Voiceprint Verification Model

In [29]:

# 2. VOICEPRINT VERIFICATION MODEL

aud_df = pd.read_csv("/content/drive/MyDrive/audio_features.csv")

voice_encoder = LabelEncoder()
y_voice = voice_encoder.fit_transform(aud_df["speaker"])

# Select only numeric acoustic features
voice_drop_cols = ["speaker", "phrase", "augmentation", "sample_rate"]
X_voice = (
    aud_df.drop(columns=[c for c in voice_drop_cols if c in aud_df.columns])
    .select_dtypes(include=[np.number])
)

print(f"Voice numeric features ({X_voice.shape[1]}): {list(X_voice.columns)}")

# split: train on 'approve', test on unseen 'transaction'
voice_train_mask = aud_df["phrase"] == "approve"
voice_test_mask = aud_df["phrase"] == "transaction"

X_voice_train, y_voice_train = X_voice[voice_train_mask], y_voice[voice_train_mask]
X_voice_test, y_voice_test = X_voice[voice_test_mask], y_voice[voice_test_mask]

voice_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42)),

])

evaluate_model(
    "2. Voiceprint Verification Model (Random Forest)",
    voice_model,
    X_voice_train, X_voice_test,
    y_voice_train, y_voice_test,
    voice_encoder.classes_,
)

# Refit on all samples
voice_model.fit(X_voice, y_voice)
joblib.dump(voice_model, "models/voice_model.joblib")
joblib.dump(voice_encoder, "models/voice_encoder.joblib")
joblib.dump(list(X_voice.columns), "models/voice_features.joblib")

Voice numeric features (32): ['sample_rate_hz', 'duration_seconds', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'spectral_rolloff_mean_hz', 'rms_energy_mean', 'spectral_centroid_mean_hz', 'zero_crossing_rate_mean']

MODEL: 2. Voiceprint Verification Model (Random Forest)
Accuracy : 0.8889
F1-Score : 0.8857 (Macro)
Log Loss : 0.6346

Classification Report:
              precision    recall  f1-score   support

     Bridget       1.00      1.00      1.00         3
       Merci       1.00      0.67      0.80         3
        Yves       0.75      1.00      0.86         3

    accuracy                           0.89         9
   macro avg       0.92      

['models/voice_features.joblib']

### Interpretation
- **Metrics:** Accuracy = **0.8889 (88.89%)** | Macro F1-Score = **0.8857** | Log Loss = **0.6346**
The classifier learned how to distinguish `'approve'` and then applied it to the transaction that was not included in the learning phase, using the `RandomForestClassifier`, and successfully verified **8 out of 9** tests.
- **Speaker Breakdown:** `Bridget` received perfect results for precision and recall, while `Merci` was misclassified **1 out of 3** times in the `"transaction"` class (recall for `Merci` is `0.67` and precision for `Yves` is `0.75`). The log loss of `0.6346` means that probabilities were averaged based on votes of trees for different spoken words on a small sample of $n=9$. All samples must be retrained before saving `models/voice_model.joblib`.

# Product Recommendation Model

In [30]:
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV

cust_df = pd.read_csv("/content/drive/MyDrive/merged_customer_data.csv")

prod_encoder = LabelEncoder()
y_prod = prod_encoder.fit_transform(cust_df["product_category"])

prod_drop_cols = [
    "customer_id_legacy", "transaction_id", "purchase_date",
    "customer_id", "customer_id_new", "product_category", "target_product",
]
X_prod = cust_df.drop(columns=[c for c in prod_drop_cols if c in cust_df.columns])

X_prod_train, X_prod_test, y_prod_train, y_prod_test = train_test_split(
    X_prod, y_prod, test_size=0.2, random_state=42, stratify=y_prod
)

# Balanced sample weights
sample_weights = compute_sample_weight(class_weight="balanced", y=y_prod_train)

# 3A. Random Forest (Baseline)
rf_prod = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
)
rf_acc, rf_f1, rf_loss = evaluate_model(
    "3A. Product Recommendation Model (Random Forest)",
    rf_prod,
    X_prod_train, X_prod_test,
    y_prod_train, y_prod_test,
    prod_encoder.classes_,
)

# 3B. Extra Trees
et_prod = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
)
et_acc, et_f1, et_loss = evaluate_model(
    "3B. Product Recommendation Model (Extra Trees)",
    et_prod,
    X_prod_train, X_prod_test,
    y_prod_train, y_prod_test,
    prod_encoder.classes_,
)

candidates = [
    ("Random Forest", rf_prod, rf_f1),
    ("Extra Trees", et_prod, et_f1),
]

# 3C. Tuned XGBoost
if HAS_XGB:
    xgb_prod = XGBClassifier(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        reg_alpha=0.2,
        reg_lambda=1.0,
        eval_metric="mlogloss",
        random_state=42,
    )
    xgb_prod.fit(X_prod_train, y_prod_train, sample_weight=sample_weights)
    y_pred = xgb_prod.predict(X_prod_test)
    y_proba = xgb_prod.predict_proba(X_prod_test)

    xgb_acc = accuracy_score(y_prod_test, y_pred)
    xgb_f1 = f1_score(y_prod_test, y_pred, average="macro")
    xgb_loss = log_loss(y_prod_test, y_proba, labels=np.arange(len(prod_encoder.classes_)))

    print(f"\n{'='*60}")
    print("MODEL: 3C. Product Recommendation Model (Tuned XGBoost)")
    print(f"{'='*60}")
    print(f"Accuracy : {xgb_acc:.4f}")
    print(f"F1-Score : {xgb_f1:.4f} (Macro)")
    print(f"Log Loss : {xgb_loss:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_prod_test, y_pred, target_names=prod_encoder.classes_, zero_division=0))

    candidates.append(("XGBoost", xgb_prod, xgb_f1))

# Select best model by Macro F1-Score
best_name, best_prod_model, best_f1 = max(candidates, key=lambda item: item[2])

print(f"\nSelected Product Model for Deployment: {best_name} (Macro F1: {best_f1:.4f})")
joblib.dump(best_prod_model, "models/product_recommender.joblib")
joblib.dump(prod_encoder, "models/target_encoder.joblib")
joblib.dump(list(X_prod.columns), "models/model_features.joblib")
print("All models saved")


MODEL: 3A. Product Recommendation Model (Random Forest)
Accuracy : 0.3953
F1-Score : 0.3729 (Macro)
Log Loss : 1.4481

Classification Report:
              precision    recall  f1-score   support

       Books       0.30      0.38      0.33         8
    Clothing       0.25      0.17      0.20         6
 Electronics       0.33      0.44      0.38         9
   Groceries       0.50      0.38      0.43         8
      Sports       0.55      0.50      0.52        12

    accuracy                           0.40        43
   macro avg       0.39      0.37      0.37        43
weighted avg       0.41      0.40      0.39        43


MODEL: 3B. Product Recommendation Model (Extra Trees)
Accuracy : 0.4186
F1-Score : 0.4055 (Macro)
Log Loss : 1.5208

Classification Report:
              precision    recall  f1-score   support

       Books       0.36      0.62      0.45         8
    Clothing       0.60      0.50      0.55         6
 Electronics       0.27      0.33      0.30         9
   Groceri

### Product Recommendation Model Interpretation & Comparison

- Context for Baseline: In all 5 classes of products (`Books`, `Clothing`, `Electronics`, `Groceries`, `Sports`), randomly guessing leads to an accuracy of 20.00% and constant log loss of approximately 1.6094$.

- Model Comparison (Stratified 80/20 Split, $n_{\text{test}} = 43$):

  - Random Forest (Baseline): accuracy 0.3953 (`17/43`) | Macro F1 0.3729 | Log Loss 1.4481 (has the best probability calibration compared to other two candidates).

  - Extra Trees: accuracy 0.4186 (`18/43`) | Macro F1 0.4055 | Log Loss 1.5208 (the strongest one on `Clothing` and `Sports` with F1 `0.55`).

  - XGBoost tuned: accuracy 0.4186 (`18/43`) | Macro F1 0.4147 | Log Loss 1.4916 (**2.09×** random baseline).

- Reason for Opting for XGBoost: Using balanced sample weights along with shallow regularized decision trees (`max_depth=3`, `learning_rate=0.04`), XGBoost was found to have achieved the highest Macro F1-Score (`0.4147`) and a balanced level of performance in all categories—it was able to perform well in `Groceries` (F1 `0.59`), `Sports` (F1 `0.55`), `Clothing` (F1 `0.43`), `Books` (F1 `0.40`), and `Electronics` (F1 `0.11`) had high behavioral similarity with others.

#Mutimodal Decision Logic

### Multimodal Decision Logic
The three exported models (`face_model.joblib`, `voice_model.joblib`, `product_recommender.joblib`) operate sequentially in the CLI pipeline (`app.py`):
1. **Face recognition model:** Predicts user identity and checks $\max P_{\text{face}} \ge \tau_{\text{face}}$. Unrecognized faces are rejected immediately.
2. **Voiceprint Verification Model:** Validates the spoken authorization command, requiring both $\max P_{\text{voice}} \ge \tau_{\text{voice}}$ and cross-modal identity consistency ($\hat{y}_{\text{face}} == \hat{y}_{\text{voice}}$).
3. **Product Recommendation Model:**outputs the ranked product category probabilities for the authenticated customer.